# PEAK M-ATH — DNS & URL Suspicion Analyzer

This notebook reads all CSV files from `input/` (recursively, e.g. input/sentinelone/, input/other/) and flags potentially suspicious DNS questions and abnormal URL access, following the **PEAK** framework: **Prepare → Execute → Act → Knowledge**.

**M-ATH approach:** Multi-signal heuristic scoring (entropy, length, subdomains, risky TLD, DGA patterns, punycode, base64) with shared detection logic overlay and uncommon-country highlighting.

## How to use
1. Put one or more CSV files into `input/` or source subfolders (e.g. `input/sentinelone/`)
2. Run all cells
3. Review on-screen summaries and files generated in `output/`

In [ ]:
pass  # Placeholder (removed environment-specific output)

## PREPARE — Plan your Approach

- **Select Topic:** DNS and URL anomaly detection — surface domains and URLs that exhibit suspicious lexical or structural characteristics (MITRE ATT&CK [T1071.001](https://attack.mitre.org/techniques/T1071/001/) Web Protocols, [T1568](https://attack.mitre.org/techniques/T1568/) Dynamic Resolution).
- **Research Topic:** Shannon entropy for randomness detection, risky TLD databases, DGA-like patterns, punycode/base64 obfuscation, uncommon-country TLD analysis.
- **Identify Datasets:** DNS and URL logs from EDR/SIEM exports; exclusion configs for known benign domains.
- **Select Algorithms:** Rule-based multi-signal scoring (entropy, length, subdomains, risky TLD, DGA patterns, punycode, base64) combined with shared detection logic (`apply_dns_logics`) and exclusion overlay.

In [ ]:
# Scenario mode: run from scenarios/dns_url_anomaly_analysis/
# Adds repo root to path and uses this scenario's input/output
import os
import sys
from pathlib import Path
SCENARIO_DIR = Path('.').resolve()
REPO_ROOT = SCENARIO_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)  # for detection_logics, exclusions
INPUT_DIR = SCENARIO_DIR / 'input'
OUTPUT_DIR = SCENARIO_DIR / 'output'
EXCLUSIONS_DIR = REPO_ROOT / 'exclusions'
SCENARIO_MODE = True
from kpi_tracker import KPITracker
tracker = KPITracker(scenario_name='dns_url_anomaly_analysis', input_dir=INPUT_DIR)

In [ ]:
from pathlib import Path
from urllib.parse import urlparse, parse_qs
import glob
import ipaddress
import math
import os
import re

import pandas as pd

from detection_logics import apply_dns_logics, apply_url_logics

pd.set_option('display.max_colwidth', 180)

if not globals().get('SCENARIO_MODE', False):
    INPUT_DIR = Path('input')
    OUTPUT_DIR = Path('output')
    EXCLUSIONS_DIR = Path('exclusions')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCLUDED_VALUES_FILE = EXCLUSIONS_DIR / 'excluded_values.conf'
EXCLUDED_VALUE_REASONS_FILE = EXCLUSIONS_DIR / 'excluded_values+reasons.conf'

def _rel(p):
    try:
        if globals().get('SCENARIO_MODE', False) and 'REPO_ROOT' in globals() and hasattr(p, 'is_relative_to') and p.is_relative_to(REPO_ROOT):
            return p.relative_to(REPO_ROOT)
        return p
    except (ValueError, AttributeError):
        return p

SUSPICIOUS_URL_KEYWORDS = [
    'login', 'verify', 'update', 'secure', 'account', 'password', 'token',
    'cmd=', 'powershell', 'base64', 'redirect', 'free-gift', 'wallet',
    'invoice', 'pay', 'webscr', 'signin', 'confirm', 'callgirl'
]

RISKY_TLDS = {
    'zip', 'mov', 'top', 'click', 'gq', 'tk', 'work', 'support',
    'rest', 'country', 'stream', 'xin'
}

print(f'Input folder: {_rel(INPUT_DIR)}')
print(f'Output folder: {_rel(OUTPUT_DIR)}')
print(f'Exclusions file (values): {_rel(EXCLUDED_VALUES_FILE)}')
print(f'Exclusions file (value+reason): {_rel(EXCLUDED_VALUE_REASONS_FILE)}')

In [ ]:
def shannon_entropy(text: str) -> float:
    text = (text or '').strip()
    if not text:
        return 0.0
    probs = [text.count(ch) / len(text) for ch in set(text)]
    return -sum(p * math.log2(p) for p in probs)


def looks_like_domain(value: str) -> bool:
    if not value or not isinstance(value, str):
        return False
    value = value.strip().lower()
    if value.startswith(('http://', 'https://')):
        return False
    domain_regex = r'^(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,63}$'
    return bool(re.match(domain_regex, value))


def normalize_domain(value: str) -> str:
    value = (value or '').strip().lower().rstrip('.')
    value = re.sub(r'\s+', '', value)
    return value


def decode_punycode_domain(value: str) -> str:
    domain = normalize_domain(value)
    if not domain or 'xn--' not in domain:
        return ''

    decoded_labels = []
    changed = False
    for label in domain.split('.'):
        if label.startswith('xn--'):
            try:
                decoded_label = label.encode('ascii').decode('idna')
                decoded_labels.append(decoded_label)
                changed = changed or decoded_label != label
            except Exception:
                decoded_labels.append(label)
        else:
            decoded_labels.append(label)

    decoded_domain = '.'.join(decoded_labels)
    return decoded_domain if changed and decoded_domain != domain else ''


def try_decode_base64_token(token: str) -> str:
    import base64

    cleaned = token.strip()
    if len(cleaned) < 12:
        return ''
    if not re.fullmatch(r'[A-Za-z0-9_+/=-]+', cleaned):
        return ''

    normalized = cleaned.replace('-', '+').replace('_', '/')
    normalized = normalized.split('=')[0]
    padded = normalized + '=' * ((4 - len(normalized) % 4) % 4)

    try:
        raw = base64.b64decode(padded, validate=True)
        decoded = raw.decode('utf-8')
    except Exception:
        return ''

    printable_ratio = sum(ch.isprintable() for ch in decoded) / max(len(decoded), 1)
    has_letters = any(ch.isalpha() for ch in decoded)
    if printable_ratio < 0.9 or not has_letters:
        return ''

    return decoded.strip()


def decode_base64_from_text(value: str) -> str:
    text = str(value or '').strip()
    if not text:
        return ''

    candidates = re.findall(r'[A-Za-z0-9_+/=-]{12,}', text)
    seen = set()
    for token in candidates:
        if token in seen:
            continue
        seen.add(token)
        decoded = try_decode_base64_token(token)
        if decoded:
            return decoded

    return ''


def decode_value_artifacts(value: str) -> str:
    punycode_decoded = decode_punycode_domain(value)
    base64_decoded = decode_base64_from_text(value)

    decoded_parts = []
    if punycode_decoded:
        decoded_parts.append(f'punycode:{punycode_decoded}')
    if base64_decoded:
        decoded_parts.append(f'base64:{base64_decoded}')

    return ' | '.join(decoded_parts)


def score_dns(domain: str):
    d = normalize_domain(domain)
    reasons = []
    score = 0

    if not looks_like_domain(d):
        return 0, ['not-domain']

    labels = d.split('.')
    joined = ''.join(labels)
    entropy = shannon_entropy(joined)

    if len(d) > 55:
        score += 2
        reasons.append('very-long-domain')
    elif len(d) > 40:
        score += 1
        reasons.append('long-domain')

    if len(labels) >= 5:
        score += 1
        reasons.append('many-subdomains')

    max_label_len = max(len(x) for x in labels)
    if max_label_len > 25:
        score += 1
        reasons.append('long-label')

    if entropy > 4.0:
        score += 2
        reasons.append('high-entropy')
    elif entropy > 3.6:
        score += 1
        reasons.append('medium-entropy')

    if 'xn--' in d:
        score += 1
        reasons.append('punycode')

    if '--' in d:
        score += 1
        reasons.append('double-hyphen')

    if labels[-1] in RISKY_TLDS:
        score += 1
        reasons.append('risky-tld')

    if re.search(r'[a-z]{8,}\d{3,}|\d{5,}[a-z]{4,}', joined):
        score += 2
        reasons.append('dga-like-pattern')

    return score, reasons


def is_ip_host(host: str) -> bool:
    try:
        ipaddress.ip_address(host)
        return True
    except Exception:
        return False


def score_url(url: str):
    raw = (url or '').strip()
    reasons = []
    score = 0

    if not raw:
        return 0, ['empty']

    parsed = urlparse(raw if '://' in raw else f'http://{raw}')
    host_parse_error = False
    try:
        host = (parsed.hostname or '').lower()
    except ValueError:
        host = ''
        host_parse_error = True
    path = parsed.path or ''
    query = parsed.query or ''
    full = raw.lower()

    if is_ip_host(host):
        score += 2
        reasons.append('ip-host')

    if len(raw) > 140:
        score += 2
        reasons.append('very-long-url')
    elif len(raw) > 100:
        score += 1
        reasons.append('long-url')

    if len(query) > 90:
        score += 1
        reasons.append('long-query')

    if raw.count('%') >= 6:
        score += 1
        reasons.append('heavy-encoding')

    params = parse_qs(query)
    if len(params) >= 8:
        score += 1
        reasons.append('many-params')

    try:
        parsed_port = parsed.port
    except ValueError:
        parsed_port = None
        host_parse_error = True

    if parsed_port and parsed_port not in {80, 443, 8080}:
        score += 1
        reasons.append('uncommon-port')

    if host_parse_error:
        reasons.append('invalid-host-format')

    if host and '.' in host:
        tld = host.split('.')[-1]
        if tld in RISKY_TLDS:
            score += 1
            reasons.append('risky-tld')

    keyword_hits = [k for k in SUSPICIOUS_URL_KEYWORDS if k in full]
    if keyword_hits:
        score += min(2, len(keyword_hits))
        reasons.append('suspicious-keywords')

    if '@' in raw:
        score += 1
        reasons.append('at-symbol-in-url')

    if re.search(r'(?:https?://)?[a-z0-9-]+(?:\.[a-z0-9-]+){4,}', full):
        score += 1
        reasons.append('deep-hostname')

    return score, reasons

## EXECUTE — Experimentation Time

- **Gather Data:** Recursively load CSVs from `input/`, detect DNS and URL columns.
- **Pre-Process Data:** Normalize domains/URLs, identify column types, load exclusion files.
- **Apply:** Score each DNS/URL value with heuristic signals and shared detection logic; filter by exclusions.
- **Analyze:** Review top findings, uncommon-country TLD analysis, and high-confidence subset (score >= 4).
- **Escalate Critical Findings:** High-score domains with DGA-like patterns or punycode obfuscation warrant immediate investigation.

In [ ]:
csv_paths = sorted(glob.glob(str(INPUT_DIR / '**' / '*.csv'), recursive=True))
print(f'Found {len(csv_paths)} CSV file(s).')
for p in csv_paths[:20]:
    print(' -', _rel(Path(p)))

if not csv_paths:
    print('No CSV files found in input/. Add files and re-run this cell.')

In [ ]:
DNS_COL_HINTS = ['dns', 'query', 'qname', 'domain', 'fqdn', 'host', 'hostname', 'request']
URL_COL_HINTS = ['url', 'uri', 'link', 'website', 'destination', 'dest', 'referrer', 'path']
COUNTRY_COL_HINTS = ['country', 'src_country', 'source_country', 'client_country', 'geo_country', 'nation', 'location_country']

def choose_columns(columns, hints):
    selected = []
    for c in columns:
        c_low = c.lower()
        if any(h in c_low for h in hints):
            selected.append(c)
    return selected


def normalize_country(value):
    text = str(value or '').strip()
    if not text or text.lower() in {'nan', 'none', 'null'}:
        return 'Unknown'
    text = re.sub(r'\s+', ' ', text)
    return text.upper() if len(text) == 2 else text.title()


def get_country_from_row(row, country_cols):
    for col_name in country_cols:
        if col_name in row.index:
            country_val = normalize_country(row[col_name])
            if country_val != 'Unknown':
                return country_val
    return 'Unknown'


def normalize_exclusion_value(value):
    return str(value or '').strip().lower()


def read_conf_lines(path: Path):
    if not path.exists():
        return []

    lines = []
    with path.open('r', encoding='utf-8') as handle:
        for raw in handle:
            line = raw.split('#', 1)[0].strip()
            if not line:
                continue
            lines.append(line)
    return lines


def load_exclusions_from_files(values_file: Path, value_reason_file: Path):
    excluded_values = {
        normalize_exclusion_value(v)
        for v in read_conf_lines(values_file)
        if normalize_exclusion_value(v)
    }

    excluded_pairs = set()
    for line in read_conf_lines(value_reason_file):
        if '||' not in line:
            continue
        value_part, reason_part = line.split('||', 1)
        v_norm = normalize_exclusion_value(value_part)
        r_norm = str(reason_part or '').strip().lower()
        if v_norm and r_norm:
            excluded_pairs.add((v_norm, r_norm))

    return excluded_values, excluded_pairs


def is_excluded(value, reasons, excluded_values, excluded_pairs):
    value_norm = normalize_exclusion_value(value)
    reason_list = [str(r or '').strip().lower() for r in reasons]

    if value_norm in excluded_values:
        return True

    for reason in reason_list:
        if (value_norm, reason) in excluded_pairs:
            return True

    return False


excluded_values_cfg, excluded_value_reason_cfg = load_exclusions_from_files(
    EXCLUDED_VALUES_FILE,
    EXCLUDED_VALUE_REASONS_FILE
)

dns_records = []
url_records = []
excluded_dns_count = 0
excluded_url_count = 0

for path in csv_paths:
    try:
        df = pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='latin-1', low_memory=False)
    except Exception as ex:
        print(f'Skipping {_rel(Path(path))}: {ex}')
        continue

    if df.empty:
        continue

    if 'tracker' in globals():
        tracker.record_rows(len(df))

    df = df.reset_index().rename(columns={'index': 'row_index'})
    columns = [str(c) for c in df.columns]
    dns_cols = choose_columns(columns, DNS_COL_HINTS)
    url_cols = choose_columns(columns, URL_COL_HINTS)
    country_cols = choose_columns(columns, COUNTRY_COL_HINTS)

    for col in dns_cols:
        series = df[col].astype(str).fillna('')
        for i, value in series.items():
            v = normalize_domain(value)
            if not looks_like_domain(v):
                continue
            score, reasons = score_dns(v)
            decoded_value = decode_value_artifacts(v)
            vt_verdict = str(df.at[i, 'vt_verdict']).strip().lower() if 'vt_verdict' in df.columns else ''
            if vt_verdict:
                decoded_value = f'{decoded_value}|vt:{vt_verdict}' if decoded_value else f'vt:{vt_verdict}'
            delta, logic_reasons = apply_dns_logics(v, decoded_value)
            score += delta
            reasons.extend(logic_reasons)
            if score >= 2:
                if is_excluded(v, reasons, excluded_values_cfg, excluded_value_reason_cfg):
                    excluded_dns_count += 1
                    continue

                row = df.loc[i]
                country = get_country_from_row(row, country_cols)
                dns_records.append({
                    'file': Path(path).name,
                    'row_index': int(row['row_index']),
                    'column': col,
                    'country': country,
                    'value': v,
                    'decoded_value': decoded_value,
                    'is_punycode': 'xn--' in v,
                    'score': score,
                    'reasons': ','.join(reasons)
                })

    for col in url_cols:
        series = df[col].astype(str).fillna('')
        for i, value in series.items():
            value = value.strip()
            if not value:
                continue
            if not (value.startswith(('http://', 'https://')) or '.' in value):
                continue
            score, reasons = score_url(value)
            decoded_value = decode_value_artifacts(value)
            vt_verdict = str(df.at[i, 'vt_verdict']).strip().lower() if 'vt_verdict' in df.columns else ''
            if vt_verdict:
                decoded_value = f'{decoded_value}|vt:{vt_verdict}' if decoded_value else f'vt:{vt_verdict}'
            delta, logic_reasons = apply_url_logics(value, decoded_value)
            score += delta
            reasons.extend(logic_reasons)
            if score >= 2:
                if is_excluded(value, reasons, excluded_values_cfg, excluded_value_reason_cfg):
                    excluded_url_count += 1
                    continue

                row = df.loc[i]
                country = get_country_from_row(row, country_cols)
                parsed_value = urlparse(value if '://' in value else f'http://{value}')
                try:
                    host = (parsed_value.hostname or '').lower()
                except ValueError:
                    host = ''
                url_records.append({
                    'file': Path(path).name,
                    'row_index': int(row['row_index']),
                    'column': col,
                    'country': country,
                    'value': value,
                    'decoded_value': decoded_value,
                    'is_punycode': 'xn--' in host or 'xn--' in value.lower(),
                    'score': score,
                    'reasons': ','.join(reasons)
                })

dns_findings = pd.DataFrame(dns_records).sort_values(['score'], ascending=False) if dns_records else pd.DataFrame(columns=['file','row_index','column','country','value','decoded_value','is_punycode','score','reasons'])
url_findings = pd.DataFrame(url_records).sort_values(['score'], ascending=False) if url_records else pd.DataFrame(columns=['file','row_index','column','country','value','decoded_value','is_punycode','score','reasons'])

if not dns_findings.empty:
    dns_findings['country'] = dns_findings['country'].fillna('Unknown')

if not url_findings.empty:
    url_findings['country'] = url_findings['country'].fillna('Unknown')

print(f'Exclusion files loaded: values={len(excluded_values_cfg)}, value+reason={len(excluded_value_reason_cfg)}')
print(f'Excluded records: DNS={excluded_dns_count}, URL={excluded_url_count}')
print(f'DNS findings: {len(dns_findings)}')
print(f'URL findings: {len(url_findings)}')

In [ ]:
print('Top suspicious DNS questions:')
display(dns_findings.head(30))

print('Top suspicious URL accesses:')
display(url_findings.head(30))

if not dns_findings.empty:
    print('DNS findings by file:')
    display(dns_findings.groupby('file', as_index=False).size().sort_values('size', ascending=False))

if not url_findings.empty:
    print('URL findings by file:')
    display(url_findings.groupby('file', as_index=False).size().sort_values('size', ascending=False))

## Uncommon country detection
This section highlights countries with unusual DNS/URL activity based on low activity share and rare punycode usage, then lists related findings for analyst review.

In [ ]:
MIN_COUNTRY_FINDINGS = 3
MAX_COUNTRY_SHARE = 0.03
RARE_PUNYCODE_MAX_HITS = 2

def summarize_uncommon_countries(findings: pd.DataFrame):
    if findings.empty or 'country' not in findings.columns:
        return pd.DataFrame(), pd.DataFrame()

    base = findings.copy()
    base['country'] = base['country'].fillna('Unknown')
    base = base[base['country'] != 'Unknown']

    if base.empty:
        return pd.DataFrame(), pd.DataFrame()

    stats = (
        base.groupby('country', as_index=False)
        .agg(
            findings_count=('country', 'size'),
            avg_score=('score', 'mean'),
            punycode_hits=('is_punycode', 'sum')
        )
        .sort_values('findings_count', ascending=False)
    )

    total_findings = stats['findings_count'].sum()
    stats['country_share'] = stats['findings_count'] / total_findings
    stats['uncommon_by_volume'] = (stats['findings_count'] <= MIN_COUNTRY_FINDINGS) | (stats['country_share'] <= MAX_COUNTRY_SHARE)
    stats['rare_punycode_country'] = (stats['punycode_hits'] > 0) & (stats['punycode_hits'] <= RARE_PUNYCODE_MAX_HITS)

    stats['flag_reason'] = stats.apply(
        lambda row: ';'.join(
            r for r in [
                'low-country-volume' if row['uncommon_by_volume'] else '',
                'rare-punycode-usage' if row['rare_punycode_country'] else ''
            ] if r
        ),
        axis=1
    )

    uncommon_stats = stats[stats['flag_reason'] != ''].copy().sort_values(
        ['rare_punycode_country', 'findings_count', 'avg_score'],
        ascending=[False, False, False]
    )

    flagged_countries = set(uncommon_stats['country'])
    uncommon_details = (
        base[base['country'].isin(flagged_countries)]
        .sort_values(['score', 'is_punycode'], ascending=[False, False])
        .copy()
    )

    return uncommon_stats, uncommon_details


uncommon_dns_countries, uncommon_dns_findings = summarize_uncommon_countries(dns_findings)
uncommon_url_countries, uncommon_url_findings = summarize_uncommon_countries(url_findings)

print('Uncommon country summary (DNS):')
display(uncommon_dns_countries.head(30))

print('Uncommon country summary (URL):')
display(uncommon_url_countries.head(30))

print('Sample DNS findings from uncommon countries:')
display(uncommon_dns_findings.head(30))

print('Sample URL findings from uncommon countries:')
display(uncommon_url_findings.head(30))

## ACT — Wrapping Up the Investigation

- **Document Findings:** HTML KPI dashboard, bar charts, and summary tables visualize hunt outcomes inline.
- **Preserve Hunt:** All findings, uncommon-country summaries, and high-confidence slices archived to `output/` CSVs.
- **Create Detections / Playbooks:** High-confidence findings (score >= 4) can feed automated detection rules or Risk-Based Alerting notables.

## Results dashboard
KPI summary and charts from the most recent execution of this notebook.

In [ ]:
from IPython.display import HTML, display
import html

def as_int(value):
    return int(value) if pd.notna(value) else 0

def pct(numerator, denominator):
    if not denominator:
        return 0.0
    return round((numerator / denominator) * 100, 2)

def build_kpi_cards(kpis):
    cards = []
    for title, value, subtitle in kpis:
        cards.append(
            f"""
            <div style='flex:1;min-width:180px;padding:12px 14px;border:1px solid #ddd;border-radius:10px;background:#fafafa;'>
                <div style='font-size:12px;color:#666'>{html.escape(title)}</div>
                <div style='font-size:26px;font-weight:700;color:#1f2937;margin-top:4px'>{html.escape(str(value))}</div>
                <div style='font-size:11px;color:#777;margin-top:2px'>{html.escape(subtitle)}</div>
            </div>
            """
        )
    return "<div style='display:flex;gap:12px;flex-wrap:wrap;margin:8px 0 14px 0'>" + ''.join(cards) + "</div>"

def render_bar_chart(df, label_col, value_col, title, max_rows=10, color='#2563eb'):
    if df is None or df.empty or label_col not in df.columns or value_col not in df.columns:
        display(HTML(f"<h4>{html.escape(title)}</h4><div>No data</div>"))
        return

    data = df[[label_col, value_col]].copy().head(max_rows)
    data[value_col] = pd.to_numeric(data[value_col], errors='coerce').fillna(0)
    max_value = max(float(data[value_col].max()), 1.0)

    rows_html = []
    for _, row in data.iterrows():
        label = str(row[label_col])
        value = float(row[value_col])
        width = max(2, int((value / max_value) * 100))
        rows_html.append(
            f"""
            <div style='display:flex;align-items:center;gap:10px;margin:6px 0'>
                <div style='width:220px;max-width:220px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;font-size:12px'>{html.escape(label)}</div>
                <div style='flex:1;background:#f3f4f6;height:14px;border-radius:7px;position:relative'>
                    <div style='width:{width}%;background:{color};height:14px;border-radius:7px'></div>
                </div>
                <div style='width:56px;text-align:right;font-size:12px'>{int(value)}</div>
            </div>
            """
        )

    chart_html = f"""
    <div style='border:1px solid #ddd;border-radius:10px;padding:10px 12px;margin:10px 0;background:#fff'>
        <div style='font-size:14px;font-weight:700;margin-bottom:6px'>{html.escape(title)}</div>
        {''.join(rows_html)}
    </div>
    """
    display(HTML(chart_html))

def render_exclusion_tables(excluded_values, excluded_pairs, max_rows=20):
    display(HTML('<h4 style="margin:10px 0 6px 0">Exclusion Rules</h4>'))

    if excluded_values:
        values_df = pd.DataFrame({'value': sorted(excluded_values)})
        print('Excluded values:')
        display(values_df.head(max_rows))
    else:
        print('Excluded values: none')

    if excluded_pairs:
        pairs_sorted = sorted(excluded_pairs, key=lambda x: (x[0], x[1]))
        pairs_df = pd.DataFrame(pairs_sorted, columns=['value', 'reason'])
        print('Excluded value + reason rules:')
        display(pairs_df.head(max_rows))
    else:
        print('Excluded value + reason rules: none')

total_dns = len(dns_findings)
total_url = len(url_findings)
total_findings = total_dns + total_url

dns_punycode = as_int(dns_findings['is_punycode'].sum()) if not dns_findings.empty else 0
url_punycode = as_int(url_findings['is_punycode'].sum()) if not url_findings.empty else 0
total_punycode = dns_punycode + url_punycode

decoded_dns = as_int((dns_findings['decoded_value'].fillna('') != '').sum()) if 'decoded_value' in dns_findings.columns else 0
decoded_url = as_int((url_findings['decoded_value'].fillna('') != '').sum()) if 'decoded_value' in url_findings.columns else 0
total_decoded = decoded_dns + decoded_url

excluded_dns = as_int(globals().get('excluded_dns_count', 0))
excluded_url = as_int(globals().get('excluded_url_count', 0))
active_excluded_values = sorted(globals().get('excluded_values_cfg', set()))
active_excluded_pairs = sorted(globals().get('excluded_value_reason_cfg', set()))

kpi_values = [
    ('Input CSV Files', len(csv_paths), 'files scanned'),
    ('Total Findings', total_findings, 'DNS + URL findings'),
    ('DNS Findings', total_dns, f"{pct(total_dns, total_findings)}% of total" if total_findings else '0% of total'),
    ('URL Findings', total_url, f"{pct(total_url, total_findings)}% of total" if total_findings else '0% of total'),
    ('Punycode Findings', total_punycode, f"{pct(total_punycode, total_findings)}% of total" if total_findings else '0% of total'),
    ('Decoded Values', total_decoded, f"{pct(total_decoded, total_findings)}% decoded" if total_findings else '0% decoded'),
    ('Excluded DNS', excluded_dns, 'records removed by exclusions'),
    ('Excluded URL', excluded_url, 'records removed by exclusions')
]

display(HTML('<h3 style="margin:4px 0 6px 0">KPI Overview</h3>'))
display(HTML(build_kpi_cards(kpi_values)))
render_exclusion_tables(active_excluded_values, active_excluded_pairs)

if not dns_findings.empty:
    dns_country = dns_findings.groupby('country', as_index=False).size().sort_values('size', ascending=False)
    dns_reason = (
        dns_findings.assign(reason=dns_findings['reasons'].fillna('').str.split(','))
        .explode('reason')
    )
    dns_reason['reason'] = dns_reason['reason'].str.strip()
    dns_reason = dns_reason[dns_reason['reason'] != '']
    dns_reason = dns_reason.groupby('reason', as_index=False).size().sort_values('size', ascending=False)

    render_bar_chart(dns_country, 'country', 'size', 'Top DNS Findings by Country')
    render_bar_chart(dns_reason, 'reason', 'size', 'Top DNS Suspicion Reasons')
else:
    display(HTML('<b>No DNS findings available for charts.</b>'))

if not uncommon_dns_countries.empty:
    render_bar_chart(uncommon_dns_countries, 'country', 'findings_count', 'Uncommon Countries (DNS Findings)')
else:
    display(HTML('<b>No uncommon DNS-country data available.</b>'))

if not url_findings.empty:
    url_country = url_findings.groupby('country', as_index=False).size().sort_values('size', ascending=False)
    render_bar_chart(url_country, 'country', 'size', 'Top URL Findings by Country', color='#7c3aed')
else:
    display(HTML('<b>No URL findings available for charts.</b>'))

In [ ]:
dns_out = OUTPUT_DIR / 'suspicious_dns_findings.csv'
url_out = OUTPUT_DIR / 'suspicious_url_findings.csv'
summary_out = OUTPUT_DIR / 'analysis_summary.csv'
uncommon_dns_out = OUTPUT_DIR / 'uncommon_country_dns_findings.csv'
uncommon_url_out = OUTPUT_DIR / 'uncommon_country_url_findings.csv'
uncommon_country_summary_out = OUTPUT_DIR / 'uncommon_country_summary.csv'

def safe_write_csv(df: pd.DataFrame, target_path: Path) -> Path:
    try:
        df.to_csv(target_path, index=False)
        return target_path
    except PermissionError:
        stamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
        fallback = target_path.with_name(f"{target_path.stem}_{stamp}{target_path.suffix}")
        df.to_csv(fallback, index=False)
        print(f"[locked] {_rel(Path(target_path))} -> wrote {_rel(Path(fallback))}")
        return fallback

saved_dns_out = safe_write_csv(dns_findings, dns_out)
saved_url_out = safe_write_csv(url_findings, url_out)
saved_uncommon_dns_out = safe_write_csv(uncommon_dns_findings, uncommon_dns_out)
saved_uncommon_url_out = safe_write_csv(uncommon_url_findings, uncommon_url_out)

summary = pd.DataFrame([
    {'metric': 'input_csv_files', 'value': len(csv_paths)},
    {'metric': 'dns_findings', 'value': len(dns_findings)},
    {'metric': 'url_findings', 'value': len(url_findings)},
    {'metric': 'uncommon_dns_findings', 'value': len(uncommon_dns_findings)},
    {'metric': 'uncommon_url_findings', 'value': len(uncommon_url_findings)}
])
saved_summary_out = safe_write_csv(summary, summary_out)

uncommon_country_summary = pd.DataFrame([
    {'artifact': 'dns', 'uncommon_countries': len(uncommon_dns_countries), 'findings': len(uncommon_dns_findings)},
    {'artifact': 'url', 'uncommon_countries': len(uncommon_url_countries), 'findings': len(uncommon_url_findings)}
])
saved_uncommon_country_summary_out = safe_write_csv(uncommon_country_summary, uncommon_country_summary_out)

print('Saved outputs:')
print(' -', _rel(Path(saved_dns_out)) if saved_dns_out else saved_dns_out)
print(' -', _rel(Path(saved_url_out)) if saved_url_out else saved_url_out)
print(' -', _rel(Path(saved_uncommon_dns_out)) if saved_uncommon_dns_out else saved_uncommon_dns_out)
print(' -', _rel(Path(saved_uncommon_url_out)) if saved_uncommon_url_out else saved_uncommon_url_out)
print(' -', _rel(Path(saved_uncommon_country_summary_out)) if saved_uncommon_country_summary_out else saved_uncommon_country_summary_out)
print(' -', _rel(Path(saved_summary_out)) if saved_summary_out else saved_summary_out)

In [ ]:
STRICT_DNS_MIN_SCORE = 4
STRICT_URL_MIN_SCORE = 4

high_conf_dns = dns_findings[dns_findings['score'] >= STRICT_DNS_MIN_SCORE].copy() if not dns_findings.empty else dns_findings.copy()
high_conf_url = url_findings[url_findings['score'] >= STRICT_URL_MIN_SCORE].copy() if not url_findings.empty else url_findings.copy()

print(f'High-confidence DNS findings (score >= {STRICT_DNS_MIN_SCORE}): {len(high_conf_dns)}')
display(high_conf_dns.head(30))

print(f'High-confidence URL findings (score >= {STRICT_URL_MIN_SCORE}): {len(high_conf_url)}')
display(high_conf_url.head(30))

strict_dns_out = OUTPUT_DIR / 'suspicious_dns_findings_high_confidence.csv'
strict_url_out = OUTPUT_DIR / 'suspicious_url_findings_high_confidence.csv'
strict_summary_out = OUTPUT_DIR / 'analysis_summary_high_confidence.csv'

high_conf_dns.to_csv(strict_dns_out, index=False)
high_conf_url.to_csv(strict_url_out, index=False)

strict_summary = pd.DataFrame([
    {'metric': 'strict_dns_min_score', 'value': STRICT_DNS_MIN_SCORE},
    {'metric': 'strict_url_min_score', 'value': STRICT_URL_MIN_SCORE},
    {'metric': 'high_conf_dns_findings', 'value': len(high_conf_dns)},
    {'metric': 'high_conf_url_findings', 'value': len(high_conf_url)}
])
strict_summary.to_csv(strict_summary_out, index=False)

print('Saved high-confidence outputs:')
print(' -', _rel(Path(strict_dns_out)) if strict_dns_out else strict_dns_out)
print(' -', _rel(Path(strict_url_out)) if strict_url_out else strict_url_out)
print(' -', _rel(Path(strict_summary_out)) if strict_summary_out else strict_summary_out)

## KNOWLEDGE — Continuous Improvement

- **Re-Add Topics to Backlog:** New suspicious TLDs, encoding patterns, or domain families discovered during analysis become future hunt topics.
- **Communicate Findings:** Share high-confidence anomalies and uncommon-country insights with SOC leadership and network security teams.
- **Feed Improvements Back:** Update risky TLD sets, suspicious keyword lists, and exclusion configs based on confirmed true/false positives; enhance shared detection logic for newly identified patterns.
- **Measure Effectiveness:** Track finding counts, high-confidence rates, and exclusion growth across runs to assess detection quality.

In [ ]:
if 'tracker' in globals():
    tracker.stop_and_report()